<a href="https://colab.research.google.com/github/sebenemaryamashebir-cmd/lab-4-llm-decision-support/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import userdata

API_KEY = userdata.get("GROQ_API_KEY")

In [2]:
!pip install openai python-dotenv pandas matplotlib

In [3]:
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


# Section 1 — Talking to an LLM Programmatically

**Part 1.1 — Your first API call**

In [4]:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response, response.choices[0].message.content


# test call on simple question
response, answer = ask_llm("What is the capital of Ethiopia?")
print(answer)

# Token usage for this call
print(response.usage)

The capital of Ethiopia is Addis Ababa.
CompletionUsage(completion_tokens=11, prompt_tokens=48, total_tokens=59, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.172958723, prompt_time=0.001446058, completion_time=0.013838309, total_time=0.015284367)


1. What is the difference between the system and user roles? Give an example of something that belongs in each.
- The system is the instruction about how the system should act. It sets the model’s constraints and behaviour for the entire conversation.
- The user is the actual question that’s being asked for the model to respond.

Example:

- “Round every answer in two decimal places” =is a system message
- “Which country is the FIFA 2026 World Cup winner?” = is the user message.
2. What is a token, roughly? Why do API providers bill per token rather than per request?
-	Token is a chunk of text usually a word or part of a word that the model splits text into before processing it. API providers bill per token because more text the model reads the more computing power it uses. So, this means short prompts cost less than long documents, and this makes pricing fair based on the work AI performs rather than charging per request which won’t be fair.

**Part 1.2 — Temperature: the randomness dial**

In [5]:
question = "Suggest a name for a savings product for market traders in Accra."

low_temp_answers = []
high_temp_answers = []

# testing 5 calls at temperature=0.0
for i in range(5):
    _, answer = ask_llm(question, temperature=0.0)
    low_temp_answers.append(answer)

# testing 5 calls at temperature=1.2
for i in range(5):
    _, answer = ask_llm(question, temperature=1.2)
    high_temp_answers.append(answer)

# Print the answers grouped by temperature
print("Temperature = 0.0")
for i, ans in enumerate(low_temp_answers, 1):
    print(f"Response {i}: ")
    print(ans)

print("\n" + "="*50)

print("\nTemperature = 1.2")
for i, ans in enumerate(high_temp_answers, 1):
    print(f"Response {i}: ")
    print(ans)

Temperature = 0.0
Response 1: 
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders who need to manage their finances on-the-go.
5. **Sika Su**: "Sika" is the Ghanaian word for "money", and "Su" means "grow" or "increase", so this name suggests a savings product that helps traders grow their wealth.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian word for "honest" or "trustworthy", which could convey a sense of reliability and security for market traders.
7. **Traders' Trust**: This name emphasizes the

1.	What did you observe at each temperature?
-	At temperature =0.0, all the 5 responses are identical every time. Temperature 0 makes the model choose the highest-probability next token at every step, so given the same prompt we get the same output repeatedly.
-	At temperature =1.2, the 5 responses differ substantially from one another. The responses have different phrasing and different product suggestions. Higher temperature flattens the probability distribution by making the model response more randomly and creatively.

2.	For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?

For loan decision-support, I choose the low temperature, possibly 0.0. This is because:
- The same applicant information should get the same recommendation (two people with same income and credit history should receive same assessment).
-	Using high temperature could create unfair treatment between applicants.
-	Institutions like banks that are giving loans should be reliable and be trusted by customers.
-	And the goal of the loan decision system is accuracy more than creativity. The aim is not to generate more ideas, rather it’s to apply correct lending criteria across applicants.




# Section 2 — The Dataset: Loan Application Letters

In [6]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


# Section 3 — Prompt Engineering for the Decision Support System

**Part 3.1 — Component 1: Summarization**

In [7]:
SUMMARY_PROMPT_V1 = "Summarize this:"
letter_L002 = LETTERS["L002"]
letter_L006 = LETTERS["L006"]
_, v1_L002 = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{letter_L002}")
_, v1_L006 = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{letter_L006}")

print("Letter L002 summary:\n")
print(v1_L002)

print("\n" + "-"*60 + "\n")

print("Letter L006 summary:\n")
print(v1_L006)


Letter L002 summary:

Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He promises to repay the loan as soon as his business improves, likely after the festive season, but currently has no collateral to offer.

------------------------------------------------------------

Letter L006 summary:

Kofi, a 22-year-old, is seeking GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no experience or collateral, but claims to be "business-minded" and "trustworthy", and promises to repay the loan within a year when his businesses are successful.


In [8]:
SUMMARY_SYSTEM_PROMPT_V2 = (
    "You are an assistant to a microfinance loan officer. Your job is to "
    "read loan applicant letters and produce short, factual, neutral briefs. "
    "Rules:\n"
    "- Write exactly 3-5 sentences.\n"
    "- Use bullet points\n"
    "- State only facts present in the letter. Do not infer, assume, or add "
    "any detail not explicitly written by the applicant.\n"
    "- Do not edit, judge, or express opinion about the applicant.\n"
    "- Focus on: who the applicant is, what they need the loan for, the "
    "amount requested (if stated), and any repayment-relevant facts "
    "mentioned (income source, existing debts, business type, collateral "
    "or guarantor, etc.)."
    "Include:\n"
    """- Applicant (name and occupation)
    - Loan amount (amount and purpose if mentioned)
    - Financial situation (income, business, debts if mentioned)
    - Security (collateral or guarantor if mentioned)
    - Repayment-related information"""
    )
def build_summary_prompt_v2 (letter_text):
  return f" Summarize this loan appplication: \n\n{letter_text}"

_, v2_L002 = ask_llm(
    build_summary_prompt_v2(letter_L002),
    system_prompt=SUMMARY_SYSTEM_PROMPT_V2,
    temperature=0.0,
)
_, v2_L006 = ask_llm(
    build_summary_prompt_v2(letter_L006),
    system_prompt=SUMMARY_SYSTEM_PROMPT_V2,
    temperature=0.0,
)

print("=== V2 — L002 ===")
print(v2_L002)
print("\n=== V2 — L006 ===")
print(v2_L006)

=== V2 — L002 ===
Here is a brief summary of the loan application:
* Applicant: Kwame Boateng, a commercial driver
* Loan amount: GHS 25,000 to repair a trotro engine and settle personal debts
* Financial situation: Business has been slow, but expected to pick up after the festive season
* Security: No collateral available
* Repayment-related information: No specific repayment plan mentioned, but the applicant is willing to pay back when able

=== V2 — L006 ===
Here is a brief summary of the loan application:
* Applicant: Kofi, 22 years old, with no stated occupation.
* Loan amount: GHS 50,000 to start a car washing business, a provision shop, and import phones from Dubai.
* Financial situation: No income or existing business mentioned, but plans to repay the loan when the businesses are "booming".
* Security: No collateral, but the applicant claims to be "trustworthy".
* Repayment-related information: Plans to repay the loan in one year.


In [9]:
# Side-by-side comparision
print("L002\n" + "-" * 60)
print("V1:", v1_L002)
print("\nV2:", v2_L002)

print("\n\nL006\n" + "-" * 60)
print("V1:", v1_L006)
print("\nV2:", v2_L006)

L002
------------------------------------------------------------
V1: Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He promises to repay the loan as soon as his business improves, likely after the festive season, but currently has no collateral to offer.

V2: Here is a brief summary of the loan application:
* Applicant: Kwame Boateng, a commercial driver
* Loan amount: GHS 25,000 to repair a trotro engine and settle personal debts
* Financial situation: Business has been slow, but expected to pick up after the festive season
* Security: No collateral available
* Repayment-related information: No specific repayment plan mentioned, but the applicant is willing to pay back when able


L006
------------------------------------------------------------
V1: Kofi, a 22-year-old, is seeking GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai

1.	What concrete problems did V1's output have that V2 fixed? Quote examples.

L002

•	V1: Says, "He's experiencing a slow period in business but expects it to improve after the festive season and is willing to repay the loan when he can." This makes it sound like "willing to repay when he can" is a repayment plan, even though it is not.

•	V2: Clearly states "Security: No collateral available" and "Repayment-related information: No specific repayment plan mentioned, but the applicant is willing to pay back when able." This makes it clear that "no specific repayment plan mentioned" is an important fact.

L006

•	V1: Says the applicant "claims to be 'business-minded'" and "promises to repay the loan in one year when his businesses are successful." This makes the future success of the businesses sound more certain than it really is.

•	V2: Clearly labels these as the applicant's own claims by stating "Security: No collateral, but the applicant claims to be 'trustworthy'" and "Financial situation: No income or existing business mentioned but plans to repay the loan when the businesses are 'booming'." The quotation marks around "trustworthy" and "booming" make it clear these are the applicant's own words, not confirmed facts.

So, in general V2 makes use of quotation marks, the applicant’s own words and explicitly mentioning absences  fixing the issue of wrong implications that might be created from V1.

2.	Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?

- This is called hallucination. A failure mode where models generate information that is not supported by the input. And the "no invented details" is essential because loan decisions must be based only on the facts provided by the applicant. If the LLM model adds information that is not in the letter, it could lead to unfair or inaccurate decisions.

**Part 3.2 — Component 2: Structured extraction (JSON)**

In [10]:
import json
import pandas as pd

# extract prompt (schema, one worked few-shot example)
EXTRACT_PROMPT =(
    "You are a data extraction engine for a microfinance loan office. "
    "You read loan applicant letters and output ONLY a single JSON object — "
    "no prose, no explanation, no markdown code fences, nothing before or "
    "after the JSON.\n\n"
    "The JSON object must have EXACTLY these keys:\n"
    "  - applicant_name (string)\n"
    "  - amount_ghs (number)\n"
    "  - purpose (string)\n"
    "  - monthly_profit_ghs (number or null)\n"
    "  - has_collateral_or_guarantor (boolean)\n"
    "  - repayment_months (number or null)\n\n"
    "Rule: if a field is not explicitly stated in the letter, set it to null. "
    "Do not guess, estimate, or infer a value that is not written in the text. "
    "Do not invent a number because it seems plausible.\n\n"
    "Here is one worked example:\n\n"
    "Letter:\n"
    "\"\"\"\n"
    "Dear Manager,\n"
    "My name is Abena Osei. I sell kente cloth at Kejetia Market and have "
    "done so for 5 years. I would like a loan of GHS 6,000 to buy more "
    "stock ahead of the wedding season. I did not mention my monthly "
    "earnings. My cousin, a civil servant, will guarantee the loan.\n"
    "\"\"\"\n\n"
    "Correct JSON output for this example:\n"
    "{\n"
    '  "applicant_name": "Abena Osei",\n'
    '  "amount_ghs": 6000,\n'
    '  "purpose": "buy more kente cloth stock ahead of the wedding season",\n'
    '  "monthly_profit_ghs": null,\n'
    '  "has_collateral_or_guarantor": true,\n'
    '  "repayment_months": null\n'
    "}\n\n"
    "Note that monthly_profit_ghs and repayment_months are null because the "
    "letter never states them, even though the applicant does mention 5 "
    "years in business and a guarantor."

)

def build_extract_prompt(letter_text):
  return f"Extract the fields from this loan application letter:\n\n{letter_text}"

# extract field (call LLM, strip fences, handle failures)

def extract_fields(letter_text):
  _, raw_output = ask_llm(
      build_extract_prompt(letter_text),
      system_prompt=EXTRACT_PROMPT,
      temperature=0.0,
      max_tokens=300,
  )

  # strip and json fences
  cleaned = raw_output.strip()
  if cleaned.startswith("```"):
    cleaned= cleaned.strip("`")
    cleaned= cleaned.replace("json", "", 1).strip() if cleaned.lower().startswith("json") else cleaned
  try:
    return json.loads(cleaned)

  except json.JSONDecodeError as e:
    print(f"WARNING: failed to parse JSON. Error:{e}")
    print(f"Raw output was:\n{raw_output}")
    return None


# run on all six letters

extracted_rows = []

for letter_id, letter_text in LETTERS.items():
    result = extract_fields(letter_text)
    if result is not None:
        result["letter_id"] = letter_id
        extracted_rows.append(result)
    else:
        # keep a placeholder row so the letter isn't silently dropped
        extracted_rows.append({"letter_id": letter_id})

extraction_df = pd.DataFrame(extracted_rows)

# Reorder columns so letter_id comes first
cols = ["letter_id"] + [c for c in extraction_df.columns if c != "letter_id"]
extraction_df = extraction_df[cols]

extraction_df


,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for feed and 500 new layers for my poultry farm,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


1.	Why must the few-shot example NOT come from the six letters you are processing?

- Because there would leak the answers into the prompts and this would make the model memorize instead of extract.

- This would also give unfair high accuracy which doesn’t test how well the model works on the new data.

- Also, from machine learning principle training and testing data should be kept separate.

2. Why "use null, do not guess" — what did the model do without that instruction?
-	The model may invent missing information by itself (hallucinate) which is not accurate and reliable.
3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?
-	Extraction has only one correct answer, so the model should be consistent. So temperature = 0 reduces the randomness and gives accurate output each time.


**Part 3.3 — Component 3: The decision-support brief**

In [11]:
# Brief prompt (letter, extracted json)
BRIEF_SYSTEM_PROMPT = (
    "You are an assistant to a microfinance loan officer. You help the "
    "officer review loan applications by producing a structured brief. "
    "You do NOT make lending decisions -- the human loan officer makes the "
    "final decision. You must never output or imply an approval or "
    "rejection recommendation (do not say 'approve', 'reject', 'should be "
    "granted', 'should be denied', or anything equivalent).\n\n"
    "Given a loan applicant's letter and a structured JSON summary of "
    "extracted facts, produce a brief with exactly these four sections:\n\n"
    "1. Strengths -- bullet points, each grounded in something explicitly "
    "stated in the letter or JSON. Do not invent strengths.\n"
    "2. Risks / Red Flags -- bullet points identifying concerns: missing "
    "collateral, inconsistent or vague repayment plans, lack of track "
    "record, unverified claims, etc.\n"
    "3. Missing Information -- bullet points listing specific facts the "
    "officer should ask the applicant to clarify or provide (e.g. proof of "
    "income, documentation, a concrete repayment plan).\n"
    "4. Suggested Next Step -- ONE of: 'invite for interview', 'request "
    "documents', 'flag for senior review', or a similarly procedural next "
    "action. This must NOT be an approval or rejection decision.\n\n"
    "Be factual and neutral. Do not add information not present in the "
    "letter or JSON."
)

def build_brief_prompt(letter_text, extracted_json):
    return (
        f"Loan applicant letter:\n\n{letter_text}\n\n"
        f"Extracted structured data:\n\n{json.dumps(extracted_json, indent=2)}\n\n"
        f"Produce the four-section brief as instructed."
    )

def generate_brief(letter_text, extracted_json):
  _, breif = ask_llm(
      build_brief_prompt(letter_text, extracted_json),
      system_prompt=BRIEF_SYSTEM_PROMPT,
      temperature=0.0,
      max_tokens=600,
  )
  return breif


# generate briefs for all six letters
briefs = {}

for letter_id, letter_text in LETTERS.items():
    # Reuse the JSON we already extracted in Part 3.2 (extraction_df),
    # rather than re-calling the LLM -- keeps the brief grounded in the
    # SAME extracted facts we already validated, and saves API calls.
    row = extraction_df[extraction_df["letter_id"] == letter_id]

    if row.empty or row.isnull().all(axis=1).iloc[0]:
        print(f"WARNING: no extracted JSON available for {letter_id}, skipping brief.")
        briefs[letter_id] = None
        continue

    extracted_json = row.drop(columns=["letter_id"]).iloc[0].to_dict()
    briefs[letter_id] = generate_brief(letter_text, extracted_json)


# Print briefs for L001, L002, L006

for letter_id in ["L001", "L002", "L006"]:
    print("-" * 70)
    print(f"BRIEF — {letter_id}")
    print("-" * 70)
    print(briefs[letter_id])
    print()

----------------------------------------------------------------------
BRIEF — L001
----------------------------------------------------------------------
## Step 1: Strengths
The applicant has several strengths that can be noted from the letter and the extracted data:
* The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market, indicating a stable and long-standing business presence.
* The applicant has a consistent profit of GHS 900 each month from the current stall, showing a viable business operation.
* The applicant has saved GHS 2,500 through the susu scheme over two years without missing a contribution, demonstrating financial discipline and commitment to savings.
* The applicant has a guarantor, her sister, who is a teacher, providing an additional layer of security for the loan.

## Step 2: Risks / Red Flags
Some potential risks and red flags can be identified:
* The expansion into frozen foods with a deep freezer is a new venture, which may 

1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the system identify the right strengths and red flags in each?
- L003 (Strong application): The brief should highlight strong evidence such as an established registered business, consistent monthly profit (GHS 2,800), collateral (GHS 5,000 fixed deposit), a clear repayment plan, and supporting sales records. These are all indicators of a strong application.
- L006 (Weak application): The brief correctly identifies the main red flags: no collateral, no business track record, a vague repayment plan, and unverified claims of being "trustworthy." It also appropriately requests more information and suggests an interview rather than making a lending decision.

2. Why did we forbid "approve"/"reject"? One practical, one ethical reason.
- Practical reason: The LLM does not have all the information needed to make a loan decision (e.g., company policies, regulations, or additional applicant details). It should only assist, does not have power to decide.
- Ethical reason: Loan decisions can greatly affect people's lives and may introduce bias. Keeping a human loan officer responsible for the final decision ensures fairness and accountability.

**Part 3.4 — Commit your prompt templates**

Commit hash:

ef9f7bd7a4243b789b829980f895285271fb4e31

# Section 4 — Evaluation: Quality, Reliability, Appropriateness

**Part 4.1 — Extraction accuracy against gold labels**

In [12]:
FIELDS = {
    "applicant_name": str,
    "amount_ghs": float,
    "purpose": str,
    "monthly_profit_ghs": float,
    "has_collateral_or_guarantor": bool,
    "repayment_months": float,
}
GOLD_LETTERS=["L001", "L003", "L006"]

def values_match(field, gold_value, extracted_value):
  # appropriate comparison
  if gold_value is None and extracted_value is None:
    return True

  if gold_value is None or extracted_value is None:
    return False

  if field == "has_collateral_or_guarantor":
    return  bool(gold_value) == bool(extracted_value)

   # ignroing capitalization
  if field in ("application_name", "purpose"):
    return str(gold_value).strip().lower() == str(extracted_value).strip().lower()

# numeric fields
  try:
    return float(gold_value)==float(extracted_value)

  except (TypeError, ValueError):
    return False

# the comparison table
comparison_rows =[]

for field in FIELDS:
  row ={"Field": field}
  match_count =0

  for letter_id in GOLD_LETTERS:
    gold_value = GOLD[letter_id].get(field)

    extracted_row= extraction_df[extraction_df["letter_id"]==letter_id]
    extracted_value = extracted_row.iloc[0][field] if not extracted_row.empty else None

    is_match = values_match(field, gold_value, extracted_value)
    match_count +=int(is_match)

    row[letter_id] = "Ok" if is_match else "No"

  row["Accuracy"] =f"{match_count}/3"
  comparison_rows.append(row)


comparison_df = pd.DataFrame(comparison_rows).set_index("Field")
comparison_df


,L001,L003,L006,Accuracy
Field,,,,
applicant_name,No,No,No,0/3
amount_ghs,Ok,Ok,Ok,3/3
purpose,No,No,No,0/3
monthly_profit_ghs,Ok,Ok,No,2/3
has_collateral_or_guarantor,Ok,Ok,Ok,3/3
repayment_months,Ok,Ok,Ok,3/3


In [13]:
total_checks = len(FIELDS)*len(GOLD_LETTERS)
total_matches = sum(int(str(comparison_df.loc[f, "Accuracy"]).split("/")[0]) for f in FIELDS)

print(f"Overall field-level accuracy: {total_matches}/{total_checks}"
      f"({total_matches/total_checks*100:.1f}%)")

Overall field-level accuracy: 11/18(61.1%)


**Part 4.2 — Reliability: is the system consistent?**

In [14]:
letter_L004 = LETTERS["L004"]

temp0_results = []
temp1_results = []

# extract_fields()nalways calls with temperature=0.0

def extract_fields_at_temp(letter_text, temperature):
    _,raw_output = ask_llm(
        build_extract_prompt(letter_text),
        system_prompt=EXTRACT_PROMPT,
        temperature=temperature,
        max_tokens=300,
    )

    cleaned = raw_output.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.replace("json", "", 1).strip() if cleaned.lower().startswith("json") else cleaned

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"WARNING: failed to parse JSON at temperature={temperature}. Error: {e}")
        print(f"Raw output was:\n{raw_output}")
        return None


# 5 runs at temperature=0.0
print("--Temperature 0.0 --")
for i in range(5):
    result = extract_fields_at_temp(letter_L004, temperature=0.0)
    temp0_results.append(result)
    print(f"Run {i+1}: {result}")

# 5 runs at temperature=1.0
print("\n-- Temperature 1.0 --")
for i in range(5):
    result = extract_fields_at_temp(letter_L004, temperature=1.0)
    temp1_results.append(result)
    print(f"Run {i+1}: {result}")

--Temperature 0.0 --
Run 1: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers for my poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 2: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers for my poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 3: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers for my poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 4: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers for my poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 5: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'buy feed and 500 new layers for poultry farm', 'monthly_profit_

**Part 4.3 — Hallucination probing**

In [15]:
# asking questions that are not on the letter

# on the letter that has no credit score
adversarial_letter = LETTERS["L004"]

test1_question = (
    f"Here is a loan applicant's letter:\n\n{adversarial_letter}\n\n"
    f"What is the applicant's credit score?"
)

_,test1_response = ask_llm(
    test1_question,
    system_prompt=SUMMARY_SYSTEM_PROMPT_V2,  # reuse the same grounded, no-invention system prompt
    temperature=0.0,
)

print("-- TEST 1 — Asking for a detail not present in the letter --")
print(test1_response)



-- TEST 1 — Asking for a detail not present in the letter --
The letter does not mention the applicant's credit score. Here is a brief based on the provided information:
* The applicant is Yaw Owusu, a poultry farmer.
* The loan amount requested is GHS 12,000 for feed and 500 new layers for his farm.
* Yaw Owusu's financial situation includes a variable income, with up to GHS 1,500 in a good month, and he has experienced losses due to bird flu.
* His uncle has agreed to guarantee the loan with his taxi.
* The applicant proposes to repay the loan in 18 months.


In [16]:
# feed the extractor an irrelevant text (weather report)

weather_report = (
    "Accra Weather Update: Skies will be partly cloudy today with a high "
    "of 31°C and a low of 24°C. Winds from the southwest at 12 km/h. "
    "Humidity around 78%. A light chance of rain in the late afternoon, "
    "clearing by evening. Tomorrow will be sunnier with similar temperatures."
)

test2_result = extract_fields(weather_report)

print("-- TEST 2 — Extracting from an irrelevant (non-loan) text --")
print(test2_result)

-- TEST 2 — Extracting from an irrelevant (non-loan) text --
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}


In [17]:
# Record verbatim outputs + PASS/FAIL judgment

print("TEST 1 RESULT")
print("-" * 60)
print("Question asked: \"What is the applicant's credit score?\"")
print("Model output (verbatim):")
print(test1_response)
print()
print("PASS criteria: model states the letter does not mention a credit")
print("score / that this information isn't provided, WITHOUT inventing a number.")
print("FAIL criteria: model states or implies a specific credit score, or")
print("otherwise answers as if the information were available.")
print()

print("TEST 2 RESULT")
print("-" * 60)
print("Input: a weather report (no applicant, no loan content at all)")
print("Model output (verbatim):")
print(test2_result)
print()
print("PASS criteria: all fields are null and/or the model refuses to")
print("produce a fabricated applicant.")
print("FAIL criteria: any field contains an invented value (a name, an")
print("amount, etc.) despite nothing applicant-related being in the text.")

TEST 1 RESULT
------------------------------------------------------------
Question asked: "What is the applicant's credit score?"
Model output (verbatim):
The letter does not mention the applicant's credit score. Here is a brief based on the provided information:
* The applicant is Yaw Owusu, a poultry farmer.
* The loan amount requested is GHS 12,000 for feed and 500 new layers for his farm.
* Yaw Owusu's financial situation includes a variable income, with up to GHS 1,500 in a good month, and he has experienced losses due to bird flu.
* His uncle has agreed to guarantee the loan with his taxi.
* The applicant proposes to repay the loan in 18 months.

PASS criteria: model states the letter does not mention a credit
score / that this information isn't provided, WITHOUT inventing a number.
FAIL criteria: model states or implies a specific credit score, or
otherwise answers as if the information were available.

TEST 2 RESULT
------------------------------------------------------------


1. Report your extraction accuracy. Which field was hardest for the model and why?
- The extraction accuracy is the value shown in our comparison_df/overall accuracy result. The applicant_name and amount_ghs fields are expected to be among the most accurate because they are stated clearly in the letters. The purpose field is likely the hardest because the model may phrase the purpose differently from the manually written GOLD value. This means a mismatch may be caused by the strict string comparison, rather than the model misunderstanding the letter.

- The monthly_profit_ghs and repayment_months fields also require attention for letters where the information is not provided. In those cases, the correct behavior is for the model to return None rather than guess.
2. What did the reliability experiment show about temperature and production systems?
- The experiment showed that temperature = 0.0 is more reliable for extraction tasks because the same input should produce the same structured answer each time. Higher temperature can introduce unnecessary variation.

- Since loan information such as the amount, repayment period, or applicant name has one correct answer, there is no benefit from creative variation. For a production lending system, consistent outputs are important, so temperature 0.0 is the safer choice.
3. Did your system hallucinate under probing? If yes, how could the prompt (or the system design around it) reduce the risk?
- No, the system did not hallucinate in either test.

- Test 1: When asked for a credit score that was not in the letter, the model correctly said that the letter did not mention a credit score and did not invent a number.
- Test 2: When given an unrelated weather report, the model returned None for all six fields instead of inventing an applicant or loan information.

- This shows that the current prompt provides good protection against missing or irrelevant information

**Part 4.4 — Appropriateness: should this system exist?**

1. Letters L002 and L006 would likely be declined. If the bank fully automates decisions with your system, who could be unfairly harmed, and how? Consider applicants who write poorly in English but run solid businesses.
- Yes, the system could unfairly harm applicants who write poorly in English but have financially healthy businesses. If the model extracts their information incorrectly or focuses on how clearly the letter is written, their application could receive a worse assessment even though the business itself is strong. This can be seen in local businesses who might not have lingustic advantages, but run a successful business in the local community. Therefore, the system should support loan officers rather than make the final decision.

2. Loan letters contain personal data. What are the implications of sending them to a third-party API in another country? What would you check before deploying this at a real Ghanaian microfinance institution?
- Loan letters contain sensitive personal and financial information. Sending them to a third-party API in another country creates privacy, security, and data-transfer risks. Before deployment, I would check how the provider stores and uses the data, whether it uses customer data for model training, where the data is stored, how it is protected, and whether the system complies with relevant Ghanaian data-protection requirements.

3. Name TWO concrete safeguards you would build around this system in production (think: human review points, logging, appeal processes, monitoring).

* Human review:  A loan officer must review the extracted information and make the final lending decision. The system should never automatically approve or reject a loan.
* Monitoring and audit logs: Store records of the model's inputs, extracted outputs, corrections, and decisions so errors and unfair patterns can be identified and investigated. Applicants should also have a way to appeal or request human review if they believe their information was handled incorrectly.


# Section 5 — Reflection



1. **Prompting as engineering:** How is iterating on a prompt similar to and different from iterating on the model hyperparameters you tuned in Lab 3?
- Prompt iteration is similar to hyperparameter tuning because both involve changing settings, testing the results, and keeping what improves performance. The difference is that hyperparameters such as learning rate and model size change how a model learns, while prompt engineering changes how we instruct an already-trained model. Prompting is also usually faster because we do not need to retrain the model.

2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended? What single evaluation result most influenced your answer?
- I would not fully trust the system to run completely unattended, especially for making decisions. The most important result was that the adversarial tests showed the model did not hallucinate when asked for a credit score that was not in the letter and returned None for all fields when given an unrelated weather report. However, because extraction errors can still occur and lending decisions have serious consequences, a human should remain involved.
3. **Cost and scale:** Estimate (from your response.usage numbers) the tokens needed to process 1,000 applications per month. What does that imply for provider choice?
- Each application used 59 tokens in the measured API call (48 prompt + 11 completion tokens). At 1,000 applications per month, this would be approximately 59,000 tokens per month. This is a relatively small workload, so provider choice should focus not only on price but also on reliability, privacy, data protection, and model performance.
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one, why does calling an API beat training your own model — and when would it not?
- For this task, calling an API is better than training our own model because a foundation model is already trained, understands natural language, and can be used immediately with relatively little data and development time. Training our own model would require a large labelled dataset, significant computing resources, and much more time. However, training our own model could be better if we had large amounts of domain-specific data, strict privacy requirements, very high usage that makes API costs expensive, or a need for complete control over the model.
